In [25]:
"""
analysis.py
-----------
Full analysis pipeline for CGVQM TAA dataset.
Investigates whether scene features predict the direction and steepness
of quality-vs-parameter trends (alpha_weight, hist_percent).

Usage:
    python analysis.py --data path/to/dataset.json --features path/to/features.csv

Features CSV expected columns:
    scene, SI, TI, colorfulness, motion, texture, dynamic_texture
"""

import argparse
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ── Import your dataloader ───────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname("../dataset.json"))
from dataloader import load_df, PARAMS, RESOLUTIONS

# ── Config ───────────────────────────────────────────────────────────────────

FOCUS_PARAMS   = ['alpha_weight', 'hist_percent']
FEATURE_COLS = ['SI', 'TI', 'CF', 'TP', 'MV', 'DTP']
RESOLUTION_INT = [50, 71, 87, 100]
SCORE_COL      = 'score_centered'   # swap to 'score' if preferred

CMAP_DIV  = 'RdBu_r'
CMAP_SEQ  = 'viridis'
FIG_DIR   = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

In [26]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f"  Saved → {path}")
    plt.close()


def spearman_ci(x, y, n_boot=2000, ci=95):
    """Spearman r with bootstrapped confidence interval."""
    r, p = stats.spearmanr(x, y)
    rng = np.random.default_rng(42)
    boot_r = []
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        br, _ = stats.spearmanr(x[idx], y[idx])
        boot_r.append(br)
    lo, hi = np.percentile(boot_r, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return r, p, lo, hi


In [27]:

# ── Step 1: Compute slopes and sensitivity ───────────────────────────────────

def compute_curve_stats(df, score_col=SCORE_COL):
    """
    For each (scene, resolution, parameter), compute:
      - slope      : Spearman r between parameter values and quality scores
      - sensitivity: max - min quality over the tested values
    Returns a DataFrame indexed by (scene, resolution, parameter).
    """
    records = []
    for (scene, res, param), grp in df.groupby(['scene', 'resolution', 'parameter']):
        grp = grp.sort_values('value')
        vals   = grp['value'].values
        scores = grp[score_col].values
        if len(vals) < 3:
            continue
        r, _ = stats.spearmanr(vals, scores)
        sensitivity = scores.max() - scores.min()
        records.append({
            'scene':       scene,
            'resolution':  res,
            'parameter':   param,
            'slope':       r,
            'sensitivity': sensitivity,
            'n_values':    len(vals),
        })
    return pd.DataFrame(records)


# ── Step 2: Curve visualisation ──────────────────────────────────────────────

def plot_quality_curves(df, features_df, param, score_col=SCORE_COL,
                        color_feature='TI'):
    """
    Overlay all 20 quality curves per resolution for a given parameter.
    Curves are color-coded by `color_feature`.
    """
    scenes = df['scene'].unique()
    feat_vals = features_df.set_index('scene')[color_feature]
    norm = plt.Normalize(feat_vals.min(), feat_vals.max())
    cmap = plt.cm.plasma

    fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
    fig.suptitle(f'Quality curves — {param}  (colored by {color_feature})',
                 fontsize=13, fontweight='bold')

    for ax, res in zip(axes, RESOLUTION_INT):
        sub = df[(df['parameter'] == param) & (df['resolution'] == res)]
        for scene in scenes:
            s = sub[sub['scene'] == scene].sort_values('value')
            if s.empty:
                continue
            color = cmap(norm(feat_vals.get(scene, np.nan)))
            ax.plot(s['value'], s[score_col], color=color, alpha=0.7, lw=1.5)
        ax.set_title(f'{res}%')
        ax.set_xlabel('Parameter value')
        ax.set_ylabel(score_col if res == RESOLUTION_INT[0] else '')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1], label=color_feature, shrink=0.8)
    plt.tight_layout()
    savefig(f'curves_{param}.png')


# ── Step 3: Resolution × slope interaction ───────────────────────────────────

def plot_slope_vs_resolution(curve_stats, param):
    """
    Plot each scene's slope as a function of resolution.
    Reveals whether direction is consistent or flips.
    """
    sub = curve_stats[curve_stats['parameter'] == param]
    scenes = sub['scene'].unique()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Slope × Resolution — {param}', fontsize=13, fontweight='bold')

    # Left: spaghetti plot
    ax = axes[0]
    for scene in scenes:
        s = sub[sub['scene'] == scene].sort_values('resolution')
        ax.plot(s['resolution'], s['slope'], marker='o', alpha=0.6, lw=1.2)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Resolution (%)')
    ax.set_ylabel('Slope (Spearman r)')
    ax.set_title('Per-scene slopes across resolutions')
    ax.set_xticks(RESOLUTION_INT)

    # Right: boxplot
    ax2 = axes[1]
    data = [sub[sub['resolution'] == r]['slope'].values for r in RESOLUTION_INT]
    ax2.boxplot(data, labels=[f'{r}%' for r in RESOLUTION_INT])
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Resolution (%)')
    ax2.set_ylabel('Slope (Spearman r)')
    ax2.set_title('Slope distribution per resolution')

    plt.tight_layout()
    savefig(f'slope_vs_resolution_{param}.png')


# ── Weighted average slope/sensitivity across resolutions ────────────────────

def weighted_avg_slopes(curve_stats):
    """
    For each (scene, parameter), compute sensitivity-weighted mean slope
    and mean sensitivity across resolutions.
    """
    records = []
    for (scene, param), grp in curve_stats.groupby(['scene', 'parameter']):
        w = grp['sensitivity'].values
        s = grp['slope'].values
        w_sum = w.sum()
        if w_sum == 0:
            w_slope = np.mean(s)
        else:
            w_slope = np.average(s, weights=w)
        records.append({
            'scene':     scene,
            'parameter': param,
            'slope_w':   w_slope,
            'sens_mean': grp['sensitivity'].mean(),
        })
    return pd.DataFrame(records)


# ── Step 4: Feature vs slope/sensitivity correlations ────────────────────────

def plot_correlation_heatmap(avg_slopes, features_df):
    """
    Heatmap: 6 features × (slope, sensitivity) × 2 parameters.
    """
    feat = features_df.set_index('scene')[FEATURE_COLS]
    results = []

    for param in FOCUS_PARAMS:
        sub = avg_slopes[avg_slopes['parameter'] == param].set_index('scene')
        for target in ['slope_w', 'sens_mean']:
            for feat_col in FEATURE_COLS:
                common = feat.index.intersection(sub.index)
                x = feat.loc[common, feat_col].values
                y = sub.loc[common, target].values
                r, p, lo, hi = spearman_ci(x, y)
                results.append({
                    'parameter': param,
                    'target':    target,
                    'feature':   feat_col,
                    'r':         r,
                    'p':         p,
                    'ci_lo':     lo,
                    'ci_hi':     hi,
                })

    res_df = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Spearman correlations: scene features vs curve statistics',
                 fontsize=13, fontweight='bold')

    for ax, target, label in zip(
        axes,
        ['slope_w', 'sens_mean'],
        ['Weighted slope (direction)', 'Mean sensitivity (magnitude)']
    ):
        pivot = res_df[res_df['target'] == target].pivot(
            index='feature', columns='parameter', values='r'
        )
        pivot = pivot.reindex(FEATURE_COLS)

        sns.heatmap(
            pivot, ax=ax, cmap=CMAP_DIV, center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', linewidths=0.5,
            cbar_kws={'label': 'Spearman r'}
        )
        ax.set_title(label)
        ax.set_xlabel('Parameter')
        ax.set_ylabel('Scene feature')

    plt.tight_layout()
    savefig('correlation_heatmap.png')
    return res_df


# ── Step 5: Scatter plots for strong correlations ────────────────────────────

def plot_scatter_strong(avg_slopes, features_df, corr_df, threshold=0.35):
    """
    For any (feature, parameter, target) with |r| > threshold, plot scatter.
    """
    feat = features_df.set_index('scene')[FEATURE_COLS]
    strong = corr_df[corr_df['r'].abs() >= threshold]

    if strong.empty:
        print(f"  No correlations above |r| = {threshold}")
        return

    n = len(strong)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, strong.iterrows()):
        sub = avg_slopes[avg_slopes['parameter'] == row['parameter']].set_index('scene')
        common = feat.index.intersection(sub.index)
        x = feat.loc[common, row['feature']].values
        y = sub.loc[common, row['target']].values
        scenes = common.values

        ax.scatter(x, y, s=60, alpha=0.8, color='steelblue', edgecolors='white', lw=0.5)
        for xi, yi, sc in zip(x, y, scenes):
            ax.annotate(sc, (xi, yi), fontsize=5, alpha=0.6,
                        xytext=(3, 3), textcoords='offset points')

        # regression line
        m, b = np.polyfit(x, y, 1)
        xr = np.linspace(x.min(), x.max(), 100)
        ax.plot(xr, m * xr + b, color='tomato', lw=1.5, ls='--')

        target_label = 'slope' if row['target'] == 'slope_w' else 'sensitivity'
        ax.set_xlabel(row['feature'])
        ax.set_ylabel(f"{target_label} ({row['parameter']})")
        ax.set_title(
            f"{row['feature']} vs {target_label}\n"
            f"({row['parameter']})  r={row['r']:.2f}  "
            f"95% CI [{row['ci_lo']:.2f}, {row['ci_hi']:.2f}]"
        )
        ax.axhline(0, color='k', lw=0.6, ls=':')

    plt.tight_layout()
    savefig('scatter_strong_correlations.png')


# ── Step 4b: PCA on scene features ───────────────────────────────────────────

def plot_pca_features(features_df, avg_slopes):
    """
    PCA biplot of scene features.
    Points colored by alpha_weight slope.
    """
    feat = features_df.set_index('scene')[FEATURE_COLS].dropna()
    scaler = StandardScaler()
    X = scaler.fit_transform(feat.values)
    pca = PCA(n_components=2)
    Z = pca.fit_transform(X)

    # color by alpha_weight slope
    aw = avg_slopes[avg_slopes['parameter'] == 'alpha_weight'].set_index('scene')
    slopes = aw.loc[feat.index, 'slope_w'].values

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=slopes, cmap=CMAP_DIV,
                    s=80, edgecolors='white', lw=0.5,
                    vmin=-1, vmax=1)
    for i, scene in enumerate(feat.index):
        ax.annotate(scene, (Z[i, 0], Z[i, 1]), fontsize=6, alpha=0.7,
                    xytext=(3, 3), textcoords='offset points')

    # loadings
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    for j, feat_name in enumerate(FEATURE_COLS):
        ax.arrow(0, 0, loadings[j, 0], loadings[j, 1],
                 head_width=0.05, head_length=0.03,
                 fc='black', ec='black', lw=1.2)
        ax.text(loadings[j, 0] * 1.12, loadings[j, 1] * 1.12,
                feat_name, fontsize=8, ha='center')

    plt.colorbar(sc, ax=ax, label='alpha_weight slope (weighted)')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('PCA biplot — scene features\n(colored by alpha_weight slope)')
    ax.axhline(0, color='k', lw=0.4, ls=':')
    ax.axvline(0, color='k', lw=0.4, ls=':')
    plt.tight_layout()
    savefig('pca_biplot.png')

    print(f"\n  PCA explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  "
          f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

    # Feature correlation matrix
    fig2, ax2 = plt.subplots(figsize=(7, 6))
    corr_mat = pd.DataFrame(feat.values, columns=FEATURE_COLS).corr(method='spearman')
    sns.heatmap(corr_mat, ax=ax2, cmap=CMAP_DIV, center=0, vmin=-1, vmax=1,
                annot=True, fmt='.2f', linewidths=0.5)
    ax2.set_title('Spearman correlation matrix — scene features')
    plt.tight_layout()
    savefig('feature_correlation_matrix.png')


# ── Step 6: Hierarchical clustering ─────────────────────────────────────────

def plot_clustering(avg_slopes, features_df):
    """
    Cluster scenes by (alpha_weight slope, hist_percent slope).
    Overlay cluster labels on PCA biplot.
    """
    pivot = avg_slopes[avg_slopes['parameter'].isin(FOCUS_PARAMS)].pivot(
        index='scene', columns='parameter', values='slope_w'
    ).dropna()

    Z = linkage(pivot.values, method='ward')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Hierarchical clustering by slope profile', fontsize=13, fontweight='bold')

    # Dendrogram
    dendrogram(Z, labels=pivot.index.tolist(), ax=axes[0],
               leaf_rotation=90, leaf_font_size=7, color_threshold=0.7 * max(Z[:, 2]))
    axes[0].set_title('Dendrogram')
    axes[0].set_ylabel('Ward distance')

    # Scatter: alpha_weight slope vs hist_percent slope, colored by cluster
    k = 3
    labels = fcluster(Z, k, criterion='maxclust')
    palette = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

    for cl in range(1, k + 1):
        idx = labels == cl
        axes[1].scatter(
            pivot.loc[idx, 'alpha_weight'],
            pivot.loc[idx, 'hist_percent'],
            s=80, label=f'Cluster {cl}',
            color=palette[cl - 1], edgecolors='white', lw=0.5
        )
    for scene in pivot.index:
        axes[1].annotate(scene,
                         (pivot.loc[scene, 'alpha_weight'],
                          pivot.loc[scene, 'hist_percent']),
                         fontsize=5, alpha=0.6,
                         xytext=(3, 3), textcoords='offset points')

    axes[1].axhline(0, color='k', lw=0.6, ls=':')
    axes[1].axvline(0, color='k', lw=0.6, ls=':')
    axes[1].set_xlabel('alpha_weight slope')
    axes[1].set_ylabel('hist_percent slope')
    axes[1].set_title(f'Slope profile (k={k} clusters)')
    axes[1].legend()

    plt.tight_layout()
    savefig('clustering.png')

    # Print cluster membership
    for cl in range(1, k + 1):
        members = pivot.index[labels == cl].tolist()
        print(f"  Cluster {cl}: {members}")

In [28]:
print("Loading data...")
df = load_df('../dataset.json', center=True)
features_df = pd.read_csv('video_stats.csv')
print(f"  Scenes: {df['scene'].nunique()}, rows: {len(df)}")

features_df['scene'] = features_df['filename'].str.replace('.mp4', '', regex=False)
features_df = features_df.drop(columns=['filename', 'error'], errors='ignore')

Loading data...
  Scenes: 20, rows: 2240


In [15]:
# ── Step 1: Compute curve stats
print("\n[Step 1] Computing slopes and sensitivity...")
curve_stats = compute_curve_stats(df, score_col=SCORE_COL)
print(curve_stats.groupby('parameter')[['slope', 'sensitivity']].describe().round(3))

# ── Step 2: Visualize quality curves
print("\n[Step 2] Plotting quality curves...")
for param in FOCUS_PARAMS:
    plot_quality_curves(df, features_df, param, score_col=SCORE_COL,
                            color_feature='TI')


[Step 1] Computing slopes and sensitivity...
             slope                                                   \
             count   mean    std    min    25%    50%    75%    max   
parameter                                                             
alpha_weight  80.0  0.104  0.562 -0.776 -0.448  0.311  0.640  0.811   
filter_size   80.0  0.084  0.326 -0.643 -0.143  0.071  0.321  0.893   
hist_percent  80.0  0.685  0.562 -0.800  0.600  1.000  1.000  1.000   
num_samples   80.0  0.180  0.520 -1.000 -0.125  0.100  0.600  1.000   

             sensitivity                                                    
                   count   mean    std    min    25%    50%    75%     max  
parameter                                                                   
alpha_weight        80.0  3.889  4.429  0.160  1.415  2.174  5.642  24.869  
filter_size         80.0  0.142  0.224  0.000  0.005  0.065  0.177   1.009  
hist_percent        80.0  1.405  1.149  0.074  0.607  0.949  2.015   4.

In [16]:
# ── Step 3: Resolution × slope
print("\n[Step 3] Plotting slope vs resolution...")
for param in FOCUS_PARAMS:
    plot_slope_vs_resolution(curve_stats, param)

# ── Weighted average slopes (collapse across resolutions)
avg_slopes = weighted_avg_slopes(curve_stats)


[Step 3] Plotting slope vs resolution...
  Saved → figures/slope_vs_resolution_alpha_weight.png
  Saved → figures/slope_vs_resolution_hist_percent.png


In [21]:
# ── Step 4: PCA on features
print("\n[Step 4a] PCA on scene features...")
plot_pca_features(features_df, avg_slopes)



[Step 4a] PCA on scene features...
  Saved → figures/pca_biplot.png

  PCA explained variance: PC1=65.6%  PC2=18.4%
  Saved → figures/feature_correlation_matrix.png


In [22]:
# ── Step 4: Correlation heatmap
print("\n[Step 4b] Computing feature-slope correlations...")
corr_df = plot_correlation_heatmap(avg_slopes, features_df)
print("\n  Top correlations:")
print(corr_df.reindex(corr_df['r'].abs().sort_values(ascending=False).index)
        .head(10)[['parameter', 'target', 'feature', 'r', 'ci_lo', 'ci_hi']]
      .to_string(index=False))



[Step 4b] Computing feature-slope correlations...
  Saved → figures/correlation_heatmap.png

  Top correlations:
   parameter    target feature        r     ci_lo    ci_hi
hist_percent sens_mean      TP 0.618045  0.223044 0.881879
hist_percent sens_mean      SI 0.616541  0.214286 0.865118
alpha_weight sens_mean      TP 0.586466  0.179380 0.853901
alpha_weight sens_mean      SI 0.575940  0.154522 0.852816
hist_percent   slope_w      TP 0.528615  0.177474 0.754316
hist_percent   slope_w      SI 0.484940  0.094142 0.755545
alpha_weight   slope_w      MV 0.308271 -0.223407 0.759281
alpha_weight   slope_w     DTP 0.306767 -0.241901 0.735808
alpha_weight   slope_w      SI 0.299248 -0.222313 0.728392
alpha_weight   slope_w      TI 0.296241 -0.256283 0.758759


In [23]:
# ── Step 5: Scatter plots
print("\n[Step 5] Scatter plots for strong correlations...")
plot_scatter_strong(avg_slopes, features_df, corr_df, threshold=0.35)


[Step 5] Scatter plots for strong correlations...
  Saved → figures/scatter_strong_correlations.png


In [24]:
# ── Step 6: Clustering
print("\n[Step 6] Hierarchical clustering...")
plot_clustering(avg_slopes, features_df)

print("\nDone. All figures saved to ./figures/")


[Step 6] Hierarchical clustering...
  Saved → figures/clustering.png
  Cluster 1: ['fantasticvillage-open', 'resto-close']
  Cluster 2: ['abandoned', 'abandoned-flipped', 'cubetest', 'oldmine', 'resto-fwd', 'wildwest-bar', 'wildwest-behindcounter']
  Cluster 3: ['abandoned-demo', 'lightfoliage', 'lightfoliage-close', 'quarry-all', 'quarry-rocksonly', 'resto-pan', 'scifi', 'subway-lookdown', 'subway-turn', 'wildwest-store', 'wildwest-town']

Done. All figures saved to ./figures/
